In [ ]:
# === DEMO FILES INVENTORY ===
import os
import glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from IPython.display import Audio, display

# Define directories
DEMO_DIR = '/fs/scratch/<allocation>/nov_demo'
AUDIO_DIR = '/fs/scratch/<allocation>/Audio Files'

# Set global variables for analysis
SAMPLE_RATE = 200000  # Default sample rate for binary files

# Quick file inventory
demo_files = sorted(glob.glob(os.path.join(DEMO_DIR, '*')))
audio_files = sorted(glob.glob(os.path.join(AUDIO_DIR, '*')))
real_files = [f for f in demo_files if f.endswith('_real.bin')]

print(f"Found {len(real_files)} .real files and {len(audio_files)} audio files")


In [ ]:
# === FILE SELECTION LIST ===
# Edit this list to choose which files to analyze
# Format: [(real_file_path, mp3_file_path), ...]

# Get available files
demo_files = sorted(glob.glob('/fs/scratch/<allocation>/nov_demo/*_real.bin'))
audio_files = sorted(glob.glob('/fs/scratch/<allocation>/Audio Files/*.mp3'))

# EDIT THIS LIST - Add/remove file pairs as needed
SELECTED_FILES = [
    ('/fs/scratch/<allocation>/nov_demo/stepped_amp2kHz_97_real.bin', '/fs/scratch/<allocation>/Audio Files/amp_2khz.mp3'),
    ('/fs/scratch/<allocation>/nov_demo/chirp_97_real.bin', '/fs/scratch/<allocation>/Audio Files/chirp_sweep.mp3'),
    ('/fs/scratch/<allocation>/nov_demo/stepped_tones_97_real.bin', '/fs/scratch/<allocation>/Audio Files/stepped_tones_sweep.mp3'),
    ('/fs/scratch/<allocation>/nov_demo/cont_amp2kHz_97_real.bin', '/fs/scratch/<allocation>/Audio Files/continuous_amp_2khz.mp3')
]

# Binary-only files (no MP3 pair needed)
BINARY_ONLY_FILES = [
    '/fs/scratch/<allocation>/nov_demo/idle_97_real.bin'
]

# Comment out auto-populate section since we're manually selecting specific files
# Auto-populate with all available pairs (you can edit this)
# Uncomment the lines below if you want to auto-populate instead of manual selection
# for real_file in demo_files:
#     real_name = os.path.basename(real_file).replace('_real.bin', '')
#     # Look for matching MP3
#     for mp3_file in audio_files:
#         mp3_name = os.path.basename(mp3_file).replace('.mp3', '')
#         if real_name in mp3_name or mp3_name in real_name:
#             SELECTED_FILES.append((real_file, mp3_file))
#             break

# Create combined list for processing
ALL_ANALYSIS_FILES = SELECTED_FILES.copy()  # File pairs
ALL_BINARY_FILES = [real for real, mp3 in SELECTED_FILES] + BINARY_ONLY_FILES  # All binary files

# Display selected files
print("🎯 SELECTED FILES FOR ANALYSIS:")
for i, (real_file, mp3_file) in enumerate(SELECTED_FILES, 1):
    print(f"{i}. {os.path.basename(real_file)} ↔ {os.path.basename(mp3_file)}")

if BINARY_ONLY_FILES:
    print(f"\n📁 BINARY-ONLY FILES (no MP3 pair):")
    for i, real_file in enumerate(BINARY_ONLY_FILES, 1):
        print(f"{i}. {os.path.basename(real_file)} (binary analysis only)")

print(f"\nTotal: {len(SELECTED_FILES)} file pairs + {len(BINARY_ONLY_FILES)} binary-only files")

# Demo Files Visualization

This notebook analyzes audio files from the demo directories for powerline interference analysis.

In [ ]:
# === TIME DOMAIN vs FREQUENCY DOMAIN COMPARISON ===
import os, glob, numpy as np, matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
try:
    import librosa
    librosa_available = True
except ImportError:
    librosa_available = False
    print("Warning: librosa not available, cannot load MP3 files")

# Use selected files from the list above
if 'SELECTED_FILES' not in globals():
    print("⚠️ Please run the FILE SELECTION cell above first")
    SELECTED_FILES = []

if librosa_available and len(SELECTED_FILES) > 0:
    # Process selected file pairs
    for idx, (real_file, mp3_file) in enumerate(SELECTED_FILES):
        real_name = os.path.basename(real_file)
        mp3_name = os.path.basename(mp3_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Load MP3 file
            mp3_data, mp3_sr = librosa.load(mp3_file, sr=None, mono=True)
            
            # Use sample rate from globals or MP3 file
            fs_real = globals().get('SAMPLE_RATE', 200000)
            fs_mp3 = mp3_sr
            
            # Create time axis for binary file
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Limit to 80 seconds
            max_time = 80
            max_samples_real = min(len(real_data), int(max_time * fs_real))
            max_samples_mp3 = min(len(mp3_data), int(max_time * fs_mp3))
            
            show_real = real_data[:max_samples_real]
            show_mp3 = mp3_data[:max_samples_mp3]
            t_real_show = t_real[:max_samples_real]
            
            # Create comparison plot: Time domain vs Frequency domain
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            
            # 1. Binary file - Time Domain
            axes[0].plot(t_real_show, show_real, 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title(f'{real_name} - Time Domain Signal')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude')
            axes[0].grid(True, alpha=0.3)
            axes[0].set_xlim(0, max_time)
            
            # 2. MP3 file - Spectrogram
            f_spec, t_spec, Sxx_mp3 = spectrogram(show_mp3, fs=fs_mp3, nperseg=1024, noverlap=512)
            im = axes[1].pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx_mp3 + 1e-12), 
                                   shading='gouraud', cmap='viridis')
            axes[1].set_title(f'{mp3_name} - Spectrogram')
            axes[1].set_xlabel('Time (s)')
            axes[1].set_ylabel('Frequency (Hz)')
            axes[1].set_ylim(0, min(25000, fs_mp3/2))
            axes[1].set_xlim(0, max_time)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error processing {real_name}/{mp3_name}: {e}')
            continue

else:
    if not librosa_available:
        print("Cannot perform comparison - librosa is required")
    else:
        print("No files selected for analysis")

In [ ]:
# === BINARY-ONLY FILES ANALYSIS (Time Domain) ===
if 'BINARY_ONLY_FILES' in globals() and len(BINARY_ONLY_FILES) > 0:
    for idx, real_file in enumerate(BINARY_ONLY_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Use sample rate from globals
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            # Create time axis for binary file
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Limit to 80 seconds
            max_time = 80
            max_samples_real = min(len(real_data), int(max_time * fs_real))
            
            show_real = real_data[:max_samples_real]
            t_real_show = t_real[:max_samples_real]
            
            # Create time domain plot
            fig, ax = plt.subplots(1, 1, figsize=(14, 6))
            
            ax.plot(t_real_show, show_real, 'b-', linewidth=0.8, alpha=0.8)
            ax.set_title(f'{real_name} - Time Domain Signal (Binary Only)')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Amplitude')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, max_time)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error processing {real_name}: {e}')
            continue

else:
    print("No binary-only files selected for analysis")

In [ ]:
# === LOW PASS FILTERING (40 Hz Cutoff) ===
from scipy.signal import butter, filtfilt

def lowpass_filter(data, cutoff, fs, order=5):
    """Apply low pass Butterworth filter"""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

if librosa_available and len(SELECTED_FILES) > 0:
    for idx, (real_file, mp3_file) in enumerate(SELECTED_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Use sample rate from globals or default
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            # Apply low pass filter with 40 Hz cutoff
            filtered_data = lowpass_filter(real_data, cutoff=65, fs=fs_real, order=5)
            
            # Create time axis
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Limit display to 80 seconds
            max_time = 80
            max_samples = min(len(real_data), int(max_time * fs_real))
            
            show_original = real_data[:max_samples]
            show_filtered = filtered_data[:max_samples]
            t_show = t_real[:max_samples]
            
            # Create comparison plot: Original vs Filtered
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            
            # 1. Original signal
            axes[0].plot(t_show, show_original, 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title(f'{real_name} - Original Signal')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude')
            axes[0].grid(True, alpha=0.3)
            axes[0].set_xlim(0, max_time)
            
            # 2. Filtered signal
            axes[1].plot(t_show, show_filtered, 'r-', linewidth=0.8, alpha=0.8)
            axes[1].set_title(f'{real_name} - After 40 Hz Low Pass Filter')
            axes[1].set_xlabel('Time (s)')
            axes[1].set_ylabel('Amplitude')
            axes[1].grid(True, alpha=0.3)
            axes[1].set_xlim(0, max_time)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error filtering {real_name}: {e}')
            continue

# Process binary-only files
if 'BINARY_ONLY_FILES' in globals() and len(BINARY_ONLY_FILES) > 0:
    for idx, real_file in enumerate(BINARY_ONLY_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Use sample rate from globals or default
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            # Apply low pass filter with 40 Hz cutoff
            filtered_data = lowpass_filter(real_data, cutoff=65, fs=fs_real, order=5)
            
            # Create time axis
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Limit display to 80 seconds
            max_time = 80
            max_samples = min(len(real_data), int(max_time * fs_real))
            
            show_original = real_data[:max_samples]
            show_filtered = filtered_data[:max_samples]
            t_show = t_real[:max_samples]
            
            # Create comparison plot: Original vs Filtered
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            
            # 1. Original signal
            axes[0].plot(t_show, show_original, 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title(f'{real_name} - Original Signal')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude')
            axes[0].grid(True, alpha=0.3)
            axes[0].set_xlim(0, max_time)
            
            # 2. Filtered signal
            axes[1].plot(t_show, show_filtered, 'r-', linewidth=0.8, alpha=0.8)
            axes[1].set_title(f'{real_name} - After 40 Hz Low Pass Filter')
            axes[1].set_xlabel('Time (s)')
            axes[1].set_ylabel('Amplitude')
            axes[1].grid(True, alpha=0.3)
            axes[1].set_xlim(0, max_time)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error filtering {real_name}: {e}')
            continue

if len(SELECTED_FILES) == 0 and (not 'BINARY_ONLY_FILES' in globals() or len(BINARY_ONLY_FILES) == 0):
    print("No files selected for filtering analysis")

In [ ]:
# === POWER SPECTRAL DENSITY (0-60 Hz) ===
from scipy.signal import welch

if librosa_available and len(SELECTED_FILES) > 0:
    for idx, (real_file, mp3_file) in enumerate(SELECTED_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Use sample rate from globals or default
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            # Apply low pass filter
            filtered_data = lowpass_filter(real_data, cutoff=65, fs=fs_real, order=5)
            
            # Compute PSD using Welch's method
            nperseg = min(len(real_data), 8192)
            
            # Original and filtered signal PSD
            f_orig, psd_orig = welch(real_data, fs=fs_real, nperseg=nperseg, noverlap=nperseg//2)
            f_filt, psd_filt = welch(filtered_data, fs=fs_real, nperseg=nperseg, noverlap=nperseg//2)
            
            # Find indices for 0-60 Hz range
            idx_60hz_orig = np.where(f_orig <= 70)[0]
            idx_60hz_filt = np.where(f_filt <= 70)[0]
            
            # Create PSD plot
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            
            # 1. Original signal PSD
            axes[0].semilogy(f_orig[idx_60hz_orig], psd_orig[idx_60hz_orig], 'b-', linewidth=1.5)
            axes[0].set_title(f'{real_name} - Original Signal PSD (0-60 Hz)')
            axes[0].set_xlabel('Frequency (Hz)')
            axes[0].set_ylabel('Power Spectral Density (V²/Hz)')
            axes[0].grid(True, alpha=0.3, which='both')
            axes[0].set_xlim(0, 70)
            
            # 2. Filtered signal PSD
            axes[1].semilogy(f_filt[idx_60hz_filt], psd_filt[idx_60hz_filt], 'r-', linewidth=1.5)
            axes[1].set_title(f'{real_name} - After 65 Hz Low Pass Filter PSD (0-60 Hz)')
            axes[1].set_xlabel('Frequency (Hz)')
            axes[1].set_ylabel('Power Spectral Density (V²/Hz)')
            axes[1].grid(True, alpha=0.3, which='both')
            axes[1].set_xlim(0, 70)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error computing PSD for {real_name}: {e}')
            continue

else:
    print("No files selected for PSD analysis")

In [ ]:
# === BANDPASS FILTERING (55-65 Hz) ===
from scipy.signal import butter, sosfiltfilt

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    """Apply bandpass Butterworth filter"""
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    sos = butter(order, [low, high], btype='band', analog=False, output='sos')
    y = sosfiltfilt(sos, data)
    return y

if librosa_available and len(SELECTED_FILES) > 0:
    for idx, (real_file, mp3_file) in enumerate(SELECTED_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                continue
            
            # Use sample rate from globals or default
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            # Apply bandpass filter (55-65 Hz)
            bandpassed_data = bandpass_filter(real_data, lowcut=55, highcut=65, fs=fs_real, order=5)
            
            # Create time axis
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Limit display to 80 seconds for full view
            max_time = 80
            max_samples = min(len(real_data), int(max_time * fs_real))
            
            show_original = real_data[:max_samples]
            show_bandpassed = bandpassed_data[:max_samples]
            t_show = t_real[:max_samples]
            
            # Also create a zoomed view (1 second) to see oscillations
            zoom_time = 1.0
            zoom_samples = int(zoom_time * fs_real)
            show_bandpassed_zoom = bandpassed_data[:zoom_samples]
            t_zoom = t_real[:zoom_samples]
            
            # Create comparison plot
            fig, axes = plt.subplots(4, 1, figsize=(14, 14))
            
            # 1. Original signal
            axes[0].plot(t_show, show_original, 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title(f'{real_name} - Original Signal')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude')
            axes[0].grid(True, alpha=0.3)
            axes[0].set_xlim(0, max_time)
            
            # 2. Bandpassed signal - Full view
            axes[1].plot(t_show, show_bandpassed, 'g-', linewidth=0.8, alpha=0.8)
            axes[1].set_title(f'{real_name} - After 55-65 Hz Bandpass Filter')
            axes[1].set_xlabel('Time (s)')
            axes[1].set_ylabel('Amplitude')
            axes[1].grid(True, alpha=0.3)
            axes[1].set_xlim(0, max_time)
            
            # 3. Bandpassed signal - Zoomed to 1 second
            axes[2].plot(t_zoom, show_bandpassed_zoom, 'g-', linewidth=1.0)
            axes[2].set_title(f'{real_name} - Bandpassed (Zoomed: 0-1 sec)')
            axes[2].set_xlabel('Time (s)')
            axes[2].set_ylabel('Amplitude')
            axes[2].grid(True, alpha=0.3)
            axes[2].set_xlim(0, zoom_time)
            
            # 4. PSD of bandpassed signal
            nperseg_bp = min(len(bandpassed_data), 8192)
            f_bp, psd_bp = welch(bandpassed_data, fs=fs_real, nperseg=nperseg_bp, noverlap=nperseg_bp//2)
            idx_bp = np.where((f_bp >= 0) & (f_bp <= 100))[0]
            
            axes[3].semilogy(f_bp[idx_bp], psd_bp[idx_bp], 'g-', linewidth=1.5)
            axes[3].axvspan(55, 65, alpha=0.2, color='green', label='Passband (55-65 Hz)')
            axes[3].set_title(f'{real_name} - PSD of Bandpassed Signal (0-100 Hz)')
            axes[3].set_xlabel('Frequency (Hz)')
            axes[3].set_ylabel('Power Spectral Density (V²/Hz)')
            axes[3].grid(True, alpha=0.3, which='both')
            axes[3].set_xlim(0, 100)
            axes[3].legend()
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error bandpass filtering {real_name}: {e}')
            continue

else:
    print("No files selected for bandpass filtering analysis")

In [ ]:
# === ENHANCED SPECTROGRAM ANALYSIS (20 kHz ± 5 kHz) ===
from scipy.signal import spectrogram, get_window

# Enhanced spectrogram parameters
FREQ_MIN_ENHANCED = 0      # Start frequency (Hz)
FREQ_MAX_ENHANCED = 6000    # End frequency (Hz)
START_TIME_SEC = 0         # Start time for analysis
DURATION_SEC = 50          # Duration for analysis

# Processing parameters for enhanced spectrograms
center_freq = 20000        # 20 kHz center frequency
bandwidth_spec = 20000     # 20 kHz total bandwidth (±10 kHz)

if librosa_available and len(ALL_BINARY_FILES) > 0:
    for idx, real_file in enumerate(ALL_BINARY_FILES):
        real_name = os.path.basename(real_file)
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                print(f'No data in {real_name}, skipping')
                continue
            
            # Use sample rate from globals
            fs_real = globals().get('SAMPLE_RATE', 200000)
            
            print(f"📊 Computing Enhanced Spectrogram for {real_name}")
            print(f"  Time range: {START_TIME_SEC}s - {START_TIME_SEC + DURATION_SEC}s")
            
            # Limit to specified duration
            max_samples_real = min(len(real_data), int(DURATION_SEC * fs_real))
            analysis_data = real_data[:max_samples_real]
            
            # Enhanced spectrogram parameters
            nperseg_spec = 4096  # Larger window for better frequency resolution
            noverlap_spec = int(nperseg_spec * 0.9)  # 90% overlap for smoother time resolution
            
            # Use Hann window to reduce spectral leakage
            window = get_window('hann', nperseg_spec)
            
            print("  Computing high-resolution spectrogram...")
            f_spec_real, t_spec_real, Sxx_real = spectrogram(analysis_data, fs=fs_real, 
                                                              nperseg=nperseg_spec, 
                                                              noverlap=noverlap_spec,
                                                              window=window)
            
            # Convert to dB
            Sxx_db = 10 * np.log10(Sxx_real + 1e-12)
            
            # Create three frequency ranges for analysis
            # 1. Low frequency (0-100 Hz) - powerline analysis
            freq_mask_low = (f_spec_real >= FREQ_MIN_ENHANCED) & (f_spec_real <= FREQ_MAX_ENHANCED)
            f_low = f_spec_real[freq_mask_low]
            Sxx_low = Sxx_db[freq_mask_low, :]
            
            # 2. High frequency (20 kHz ± 5 kHz) - enhanced analysis
            freq_min_high = center_freq - bandwidth_spec / 2  # 15 kHz
            freq_max_high = center_freq + bandwidth_spec / 2  # 25 kHz
            freq_mask_high = (f_spec_real >= freq_min_high) & (f_spec_real <= freq_max_high)
            f_high = f_spec_real[freq_mask_high]
            Sxx_high = Sxx_db[freq_mask_high, :]
            
            # Create time axis
            time_axis = np.arange(len(analysis_data)) / fs_real
            
            # Create figure with 3 subplots
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12))
            
            # Plot 1: Low frequency spectrogram (0-100 Hz)
            if len(f_low) > 0:
                vmin_low = np.percentile(Sxx_low, 3)
                vmax_low = np.percentile(Sxx_low, 97)
                
                im1 = ax1.imshow(Sxx_low, aspect='auto', origin='lower',
                                extent=[0, DURATION_SEC, f_low.min(), f_low.max()],
                                cmap='viridis', interpolation='bilinear',
                                vmin=vmin_low, vmax=vmax_low)
                
                ax1.set_title(f'{real_name} - Powerline Spectrogram (0-{FREQ_MAX_ENHANCED} Hz)', 
                              fontsize=14, fontweight='bold')
                ax1.set_ylabel('Frequency (Hz)', fontsize=12)
                ax1.set_xlim(0, DURATION_SEC)
                ax1.axhline(y=60, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='60 Hz')
                ax1.legend(loc='upper right')
                ax1.grid(True, alpha=0.2, color='white', linewidth=0.5)
                
                #plt.colorbar(im1, ax=ax1, label='Power/Frequency (dB/Hz)')
            
            # Plot 2: High frequency spectrogram (20 kHz ± 5 kHz)
            if len(f_high) > 0:
                vmin_high = np.percentile(Sxx_high, 5)
                vmax_high = np.percentile(Sxx_high, 95)
                
                im2 = ax2.imshow(Sxx_high, aspect='auto', origin='lower',
                                extent=[0, DURATION_SEC, f_high.min()/1000, f_high.max()/1000],
                                cmap='gray', interpolation='bilinear',
                                vmin=vmin_high, vmax=vmax_high)
                
                ax2.set_title(f'{real_name} - Enhanced Spectrogram at {center_freq/1000:.0f} kHz (±{bandwidth_spec/2000:.0f} kHz)', 
                              fontsize=14, fontweight='bold')
                ax2.set_ylabel('Frequency (kHz)', fontsize=12)
                ax2.set_xlim(0, DURATION_SEC)
                ax2.axhline(y=center_freq/1000, color='red', linestyle='--', linewidth=1.5, alpha=0.7, 
                           label=f'{center_freq/1000:.0f} kHz')
                ax2.legend(loc='upper right')
                ax2.grid(True, alpha=0.2, color='white', linewidth=0.5)
                
                #plt.colorbar(im2, ax=ax2, label='Power/Frequency (dB/Hz)')
            
            # Plot 3: Time domain signal
            ax3.plot(time_axis, analysis_data, color='blue', linewidth=0.5, alpha=0.8)
            ax3.set_title(f'{real_name} - Time Domain Signal ({START_TIME_SEC}-{START_TIME_SEC+DURATION_SEC}s)', 
                          fontsize=14, fontweight='bold')
            ax3.set_xlabel('Time (seconds)', fontsize=12)
            ax3.set_ylabel('Amplitude', fontsize=12)
            ax3.set_xlim(0, DURATION_SEC)
            ax3.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✓ Enhanced spectrogram analysis complete for {real_name}")
            print(f"  Low freq range: {FREQ_MIN_ENHANCED} - {FREQ_MAX_ENHANCED} Hz")
            print(f"  High freq range: {freq_min_high/1000:.1f} - {freq_max_high/1000:.1f} kHz")
            print(f"  Time duration: {DURATION_SEC} seconds")
            print(f"  Frequency resolution: {f_spec_real[1]-f_spec_real[0]:.2f} Hz")
            print(f"  Time resolution: {t_spec_real[1]-t_spec_real[0]:.3f} seconds")
            
        except Exception as e:
            print(f'Error computing enhanced spectrogram for {real_name}: {e}')
            continue
    
    print('\n🎯 Enhanced spectrogram visualization complete')
else:
    print("No files available for enhanced spectrogram analysis")

In [ ]:
# === CHIRP SIGNAL PSD ANALYSIS (0-30 kHz) with Peak Detection ===
from scipy.signal import welch, find_peaks
from scipy.optimize import curve_fit

# Target the chirp signal specifically
chirp_file = '/fs/scratch/<allocation>/May29_Alice/Chap_1_real.bin'

if os.path.exists(chirp_file):
    print(f"📊 Analyzing Chirp Signal PSD: {os.path.basename(chirp_file)}")
    
    try:
        # Load chirp binary file
        chirp_data = np.fromfile(chirp_file, dtype=np.float32)
        if len(chirp_data) == 0:
            print("❌ No data in chirp file")
        else:
            # Use sample rate from globals
            fs_chirp = globals().get('SAMPLE_RATE', 200000)
            
            # Limit analysis to first 40 seconds for consistency
            max_samples = min(len(chirp_data), int(40 * fs_chirp))
            analysis_chirp = chirp_data[:max_samples]
            
            print(f"  Sample rate: {fs_chirp:,} Hz")
            print(f"  Analysis duration: {len(analysis_chirp)/fs_chirp:.1f} seconds")
            print(f"  Total samples: {len(analysis_chirp):,}")
            
            # Compute PSD using Welch's method with high resolution
            nperseg_psd = min(len(analysis_chirp), 16384)  # Larger window for better freq resolution
            noverlap_psd = int(nperseg_psd * 0.75)
            
            print("  Computing high-resolution PSD...")
            f_psd, psd_chirp = welch(analysis_chirp, fs=fs_chirp, 
                                   nperseg=nperseg_psd, 
                                   noverlap=noverlap_psd,
                                   window='hann')
            
            # Focus on 0-5 kHz range
            freq_max_analysis = 5000  # 5 kHz
            freq_mask = (f_psd >= 0) & (f_psd <= freq_max_analysis)
            f_analysis = f_psd[freq_mask]
            psd_analysis = psd_chirp[freq_mask]
            
            # Convert to dB for better visualization
            psd_db = 10 * np.log10(psd_analysis + 1e-12)
            
            # Find peaks in the PSD
            # Use prominence and height thresholds to find significant peaks
            prominence_threshold = 5  # dB above surrounding noise
            height_threshold = np.percentile(psd_db, 70)  # Above 70th percentile
            
            peaks, peak_properties = find_peaks(psd_db, 
                                              prominence=prominence_threshold,
                                              height=height_threshold,
                                              distance=int(len(psd_db)/100))  # Minimum separation
            
            # Find global maximum
            global_peak_idx = np.argmax(psd_db)
            global_peak_freq = f_analysis[global_peak_idx]
            global_peak_power = psd_db[global_peak_idx]
            
            # Sort peaks by power (descending)
            if len(peaks) > 0:
                peak_powers = psd_db[peaks]
                peak_freqs = f_analysis[peaks]
                sorted_indices = np.argsort(peak_powers)[::-1]  # Descending order
                top_peaks = peaks[sorted_indices[:10]]  # Top 10 peaks
                top_peak_freqs = peak_freqs[sorted_indices[:10]]
                top_peak_powers = peak_powers[sorted_indices[:10]]
            
            # Create comprehensive PSD plot
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))
            
            # Plot 1: Full PSD (0-30 kHz) with peak annotations
            ax1.semilogy(f_analysis/1000, psd_analysis, 'b-', linewidth=1.0, alpha=0.8, label='Chirp PSD')
            
            # Mark global peak
            ax1.semilogy(global_peak_freq/1000, psd_analysis[global_peak_idx], 'ro', 
                        markersize=10, label=f'Global Peak: {global_peak_freq/1000:.2f} kHz')
            
            # Mark top local peaks
            if len(peaks) > 0:
                ax1.semilogy(top_peak_freqs[:5]/1000, psd_analysis[top_peaks[:5]], 'go', 
                           markersize=8, alpha=0.7, label='Top 5 Local Peaks')
                
                # Annotate top 3 peaks
                for i in range(min(3, len(top_peaks))):
                    ax1.annotate(f'{top_peak_freqs[i]/1000:.1f} kHz', 
                               xy=(top_peak_freqs[i]/1000, psd_analysis[top_peaks[i]]),
                               xytext=(10, 10), textcoords='offset points',
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                               arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
            
            ax1.set_title(f'Chirp Signal - Power Spectral Density (0-{freq_max_analysis/1000:.0f} kHz)', 
                          fontsize=14, fontweight='bold')
            ax1.set_xlabel('Frequency (kHz)', fontsize=12)
            ax1.set_ylabel('Power Spectral Density (V²/Hz)', fontsize=12)
            ax1.grid(True, alpha=0.3, which='both')
            ax1.legend()
            ax1.set_xlim(0, freq_max_analysis/1000)
            
            # Plot 2: PSD in dB scale with peak detection details
            ax2.plot(f_analysis/1000, psd_db, 'b-', linewidth=1.0, alpha=0.8, label='Chirp PSD (dB)')
            
            # Mark global peak
            ax2.plot(global_peak_freq/1000, global_peak_power, 'ro', 
                    markersize=10, label=f'Global Peak: {global_peak_freq/1000:.2f} kHz')
            
            # Mark all detected peaks
            if len(peaks) > 0:
                ax2.plot(f_analysis[peaks]/1000, psd_db[peaks], 'go', 
                        markersize=6, alpha=0.7, label=f'{len(peaks)} Detected Peaks')
                
                # Add horizontal line at detection threshold
                ax2.axhline(y=height_threshold, color='red', linestyle='--', alpha=0.5, 
                           label=f'Detection Threshold: {height_threshold:.1f} dB')
            
            ax2.set_title(f'Chirp Signal - PSD with Peak Detection (dB Scale)', 
                          fontsize=14, fontweight='bold')
            ax2.set_xlabel('Frequency (kHz)', fontsize=12)
            ax2.set_ylabel('Power Spectral Density (dB)', fontsize=12)
            ax2.grid(True, alpha=0.3)
            ax2.legend()
            ax2.set_xlim(0, freq_max_analysis/1000)
            
            plt.tight_layout()
            plt.show()
            
            # Print detailed analysis results
            print(f"\n🎯 CHIRP SIGNAL ANALYSIS RESULTS:")
            print(f"  Frequency resolution: {f_analysis[1]-f_analysis[0]:.2f} Hz")
            print(f"  Analysis range: 0 - {freq_max_analysis/1000:.0f} kHz")
            
            print(f"\n📈 GLOBAL PEAK:")
            print(f"  Frequency: {global_peak_freq:.2f} Hz ({global_peak_freq/1000:.3f} kHz)")
            print(f"  Power: {psd_analysis[global_peak_idx]:.2e} V²/Hz")
            print(f"  Power (dB): {global_peak_power:.1f} dB")
            
            if len(peaks) > 0:
                print(f"\n🔍 TOP LOCAL PEAKS ({min(10, len(peaks))} of {len(peaks)} detected):")
                for i in range(min(10, len(top_peaks))):
                    freq_hz = top_peak_freqs[i]
                    power_linear = psd_analysis[top_peaks[i]]
                    power_db = top_peak_powers[i]
                    print(f"  {i+1:2d}. {freq_hz:8.2f} Hz ({freq_hz/1000:7.3f} kHz) | "
                          f"{power_linear:.2e} V²/Hz | {power_db:6.1f} dB")
                
                # Check if this looks like a chirp sweep
                freq_range = np.max(top_peak_freqs[:5]) - np.min(top_peak_freqs[:5])
                print(f"\n📊 CHIRP CHARACTERISTICS:")
                print(f"  Frequency span of top 5 peaks: {freq_range:.0f} Hz ({freq_range/1000:.2f} kHz)")
                print(f"  Peak distribution suggests: ", end="")
                if freq_range > 1000:
                    print("Broadband chirp sweep ✓")
                elif len(peaks) > 20:
                    print("Multi-tone or complex chirp ✓")
                else:
                    print("Narrow-band or single-tone signal")
            else:
                print(f"\n⚠️  No significant peaks detected above threshold")
                
    except Exception as e:
        print(f"❌ Error analyzing chirp signal: {e}")

else:
    print(f"❌ Chirp file not found: {chirp_file}")
    print("   Available binary files:")
    if 'ALL_BINARY_FILES' in globals():
        for f in ALL_BINARY_FILES:
            print(f"   - {os.path.basename(f)}")

In [ ]:
# === ODD HARMONIC COMPONENTS - TIME DOMAIN FILTERING (60, 180, 300, 420, 540 Hz) ===
from scipy.signal import butter, sosfiltfilt

def bandpass_filter_harmonic(data, center_freq, fs, bandwidth=10, order=4):
    """Apply narrow bandpass filter around a specific harmonic frequency"""
    nyquist = 0.5 * fs
    low = (center_freq - bandwidth/2) / nyquist
    high = (center_freq + bandwidth/2) / nyquist
    sos = butter(order, [low, high], btype='band', analog=False, output='sos')
    y = sosfiltfilt(sos, data)
    return y

# Define the 5 odd harmonics (60 Hz fundamental + odd multiples)
harmonics = [60*60, 60*61, 60*62, 60*63, 60*64]
harmonic_labels = ['1st (60 Hz)', '3rd (180 Hz)', '5th (300 Hz)', '7th (420 Hz)', '9th (540 Hz)']
colors = ['red', 'green', 'orange', 'purple', 'brown']

if os.path.exists(chirp_file):
    print(f"📊 Filtering Odd Harmonic Components: {os.path.basename(chirp_file)}")
    
    try:
        # Load chirp binary file
        chirp_data = np.fromfile(chirp_file, dtype=np.float32)
        if len(chirp_data) == 0:
            print("❌ No data in chirp file")
        else:
            # Use sample rate from globals
            fs_chirp = globals().get('SAMPLE_RATE', 200000)
            
            # Limit analysis to first 40 seconds for clear visualization
            max_samples = min(len(chirp_data), int(40 * fs_chirp))
            analysis_chirp = chirp_data[:max_samples]
            
            # Create time axis
            t = np.arange(len(analysis_chirp)) / fs_chirp
            
            # Filter bandwidth (±5 Hz around each harmonic)
            filter_bandwidth = 10  # Total bandwidth of 10 Hz (±5 Hz)
            
            # Apply bandpass filter for each harmonic
            print(f"  Applying bandpass filters (±{filter_bandwidth/2:.0f} Hz) around each harmonic...")
            filtered_harmonics = []
            
            for harm_freq in harmonics:
                filtered_signal = bandpass_filter_harmonic(analysis_chirp, harm_freq, 
                                                          fs_chirp, bandwidth=filter_bandwidth, order=5)
                filtered_harmonics.append(filtered_signal)
            
            # Create comprehensive visualization
            fig, axes = plt.subplots(6, 1, figsize=(16, 18))
            
            # Plot 0: Original signal (first 1 second for detail)
            zoom_samples = int(40.0 * fs_chirp)
            t_zoom = t[:zoom_samples]
            
            axes[0].plot(t_zoom, analysis_chirp[:zoom_samples], 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title('Original Signal (0-40 seconds)', fontsize=14, fontweight='bold')
            axes[0].set_xlabel('Time (s)', fontsize=11)
            axes[0].set_ylabel('Amplitude', fontsize=11)
            axes[0].grid(True, alpha=0.3)
            
            # Plot 1-5: Each filtered harmonic component (first 1 second)
            for i, (harm_freq, label, color, filtered_sig) in enumerate(zip(harmonics, harmonic_labels, colors, filtered_harmonics)):
                axes[i+1].plot(t_zoom, filtered_sig[:zoom_samples], color=color, linewidth=1.0, alpha=0.9)
                axes[i+1].set_title(f'{label} - Bandpass Filtered ({harm_freq-filter_bandwidth/2:.0f}-{harm_freq+filter_bandwidth/2:.0f} Hz)', 
                                  fontsize=13, fontweight='bold')
                axes[i+1].set_xlabel('Time (s)', fontsize=11)
                axes[i+1].set_ylabel('Amplitude', fontsize=11)
                axes[i+1].grid(True, alpha=0.3)
                
                # Add frequency annotation
                period = 1.0 / harm_freq
                axes[i+1].text(0.98, 0.95, f'Period: {period*1000:.1f} ms', 
                             transform=axes[i+1].transAxes, 
                             ha='right', va='top',
                             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
                             fontsize=10)
            
            plt.tight_layout()
            plt.show()
            
            # Print detailed results
            print(f"\n🎯 ODD HARMONIC FILTERING RESULTS:")
            print(f"  Sample rate: {fs_chirp:,} Hz")
            print(f"  Analysis duration: {len(analysis_chirp)/fs_chirp:.1f} seconds")
            print(f"  Filter bandwidth: ±{filter_bandwidth/2:.0f} Hz around each harmonic")
            
            print(f"\n📊 FILTERED HARMONIC COMPONENTS:")
            for i, (harm_freq, label) in enumerate(zip(harmonics, harmonic_labels)):
                rms_value = np.sqrt(np.mean(filtered_harmonics[i]**2))
                peak_value = np.max(np.abs(filtered_harmonics[i]))
                print(f"  {label:20s} | Center: {harm_freq:3d} Hz | "
                      f"RMS: {rms_value:.4f} | Peak: {peak_value:.4f}")
                        
    except Exception as e:
        print(f"❌ Error filtering odd harmonics: {e}")

else:
    print(f"❌ Chirp file not found: {chirp_file}")

In [ ]:
# === COMBINED HARMONICS vs ORIGINAL COMPARISON ===

if os.path.exists(chirp_file) and 'filtered_harmonics' in globals():
    print(f"📊 Comparing Combined Harmonics with Original Signal")
    
    try:
        # Normalize each harmonic component before summing
        normalized_harmonics = []
        for i, filtered_sig in enumerate(filtered_harmonics):
            # Normalize to unit RMS (or zero if signal is all zeros)
            rms = np.sqrt(np.mean(filtered_sig**2))
            if rms > 0:
                normalized = filtered_sig / rms
            else:
                normalized = filtered_sig
            normalized_harmonics.append(normalized)
        
        # Sum all normalized harmonics
        combined_signal = np.sum(normalized_harmonics, axis=0)
        
        # Normalize the original signal for comparison
        original_rms = np.sqrt(np.mean(analysis_chirp**2))
        if original_rms > 0:
            normalized_original = analysis_chirp / original_rms
        else:
            normalized_original = analysis_chirp
        
        # Create comparison figure
        fig, axes = plt.subplots(2, 1, figsize=(16, 14))
        
        # Plot 1: Normalized original signal (full duration)
        axes[0].plot(t, normalized_original, 'b-', linewidth=0.8, alpha=0.8, label='Normalized Original')
        axes[0].set_title('Normalized Original Signal (Full Duration)', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Time (s)', fontsize=12)
        axes[0].set_ylabel('Normalized Amplitude', fontsize=12)
        axes[0].grid(True, alpha=0.3)
        axes[0].legend(loc='upper right')
        
        # Plot 2: Combined normalized harmonics (full duration)
        axes[1].plot(t, combined_signal, 'r-', linewidth=0.8, alpha=0.8, label='Combined Normalized Harmonics')
        axes[1].set_title('Combined Normalized Odd Harmonics (60+180+300+420+540 Hz)', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Time (s)', fontsize=12)
        axes[1].set_ylabel('Normalized Amplitude', fontsize=12)
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
        
        # Residual calculation
        residual = normalized_original - combined_signal
        
        # Print comparison statistics
        norm_original_rms = np.sqrt(np.mean(normalized_original**2))
        combined_rms = np.sqrt(np.mean(combined_signal**2))
        combined_peak = np.max(np.abs(combined_signal))
        residual_rms = np.sqrt(np.mean(residual**2))
        
        print(f"\n📈 NORMALIZED COMPARISON STATISTICS:")
        print(f"  Normalized Original RMS:  {norm_original_rms:.4f}")
        print(f"  Combined Harmonics RMS:   {combined_rms:.4f}")
        print(f"  Combined Peak:            {combined_peak:.4f}")
        print(f"  Residual RMS:             {residual_rms:.4f}")
        print(f"\n  Individual Normalized Harmonic RMS:")
        for i, (label, norm_harm) in enumerate(zip(harmonic_labels, normalized_harmonics)):
            harm_rms = np.sqrt(np.mean(norm_harm**2))
            print(f"    {label:20s} | RMS: {harm_rms:.4f}")
        print(f"\n  Residual energy:  {(residual_rms/norm_original_rms)*100:.1f}% of normalized original")
        
        # Play combined harmonics as audio
        print(f"\n🔊 AUDIO PLAYBACK:")
        print("  Playing combined normalized harmonics as audio...")
        
        # Normalize to prevent clipping (scale to [-0.9, 0.9] range)
        audio_signal = combined_signal / (np.max(np.abs(combined_signal)) + 1e-10) * 0.9
        
        # Display audio player
        display(Audio(audio_signal, rate=fs_chirp))
        
    except Exception as e:
        print(f"❌ Error in comparison: {e}")
else:
    print("⚠️ Please run the harmonic filtering cell first")

In [ ]:
# === ENVELOPE EXTRACTION AND AUDIO PLAYBACK ===
from scipy.signal import hilbert

if os.path.exists(chirp_file) and 'combined_signal' in globals():
    print(f"📊 Extracting Envelope from Combined Harmonics Signal")
    
    try:
        # Extract envelope using Hilbert transform
        analytic_signal = hilbert(combined_signal)
        envelope = np.abs(analytic_signal)
        
        # Create comparison figure
        fig, axes = plt.subplots(3, 1, figsize=(16, 14))
        
        # Zoom to first 5 seconds for detailed view
        zoom_samples = int(5.0 * fs_chirp)
        t_zoom = t[:zoom_samples]
        
        # Plot 1: Combined harmonics signal (zoomed)
        axes[0].plot(t_zoom, combined_signal[:zoom_samples], 'b-', linewidth=0.8, alpha=0.7, label='Combined Harmonics')
        axes[0].set_title('Combined Harmonics Signal (0-5 seconds)', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Time (s)', fontsize=12)
        axes[0].set_ylabel('Normalized Amplitude', fontsize=12)
        axes[0].grid(True, alpha=0.3)
        axes[0].legend(loc='upper right')
        
        # Plot 2: Envelope overlaid on signal (zoomed)
        axes[1].plot(t_zoom, combined_signal[:zoom_samples], 'b-', linewidth=0.5, alpha=0.4, label='Combined Harmonics')
        axes[1].plot(t_zoom, envelope[:zoom_samples], 'r-', linewidth=2.0, alpha=0.9, label='Envelope')
        axes[1].plot(t_zoom, -envelope[:zoom_samples], 'r-', linewidth=2.0, alpha=0.9)
        axes[1].set_title('Envelope Extraction (0-5 seconds)', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Time (s)', fontsize=12)
        axes[1].set_ylabel('Normalized Amplitude', fontsize=12)
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc='upper right')
        
        # Plot 3: Envelope only (full duration)
        axes[2].plot(t, envelope, 'r-', linewidth=1.0, alpha=0.9, label='Envelope')
        axes[2].set_title('Envelope Signal (Full Duration)', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Time (s)', fontsize=12)
        axes[2].set_ylabel('Envelope Amplitude', fontsize=12)
        axes[2].grid(True, alpha=0.3)
        axes[2].legend(loc='upper right')
        axes[2].fill_between(t, 0, envelope, alpha=0.3, color='red')
        
        plt.tight_layout()
        plt.show()
        
        # Calculate envelope statistics
        envelope_rms = np.sqrt(np.mean(envelope**2))
        envelope_peak = np.max(envelope)
        envelope_mean = np.mean(envelope)
        
        print(f"\n📈 ENVELOPE STATISTICS:")
        print(f"  RMS:   {envelope_rms:.4f}")
        print(f"  Peak:  {envelope_peak:.4f}")
        print(f"  Mean:  {envelope_mean:.4f}")
        
        # Normalize envelope for audio playback
        audio_envelope = envelope / (np.max(envelope) + 1e-10) * 0.9
        
        print(f"\n🔊 AUDIO PLAYBACK:")
        print("  Playing envelope as audio...")
        print(f"  Sample rate: {fs_chirp:,} Hz")
        print(f"  Duration: {len(audio_envelope)/fs_chirp:.1f} seconds")
        
        # Display audio player for envelope
        display(Audio(audio_envelope, rate=fs_chirp))
        
    except Exception as e:
        print(f"❌ Error extracting envelope: {e}")
else:
    print("⚠️ Please run the harmonic filtering and combination cells first")